# S3.4 Ingestion Pipeline

Interactive verification of the ingestion pipeline spec:
- Airflow DAG task functions (setup, fetch, report)
- FastAPI ingestion endpoint (POST /api/v1/ingest/fetch)
- Request/response schemas

In [1]:
import sys
sys.path.insert(0, "../..")
sys.path.insert(0, "../../airflow/dags")
print("\u2713 sys.path configured")

✓ sys.path configured


## 1. Constants & Task Module Imports

In [2]:
from arxiv_ingestion.common import API_BASE_URL, HEALTH_URL, INGEST_FETCH_URL, FETCH_TIMEOUT, DEFAULT_TIMEOUT

assert API_BASE_URL == "http://api:8000"
assert HEALTH_URL == "http://api:8000/api/v1/ping"
assert INGEST_FETCH_URL == "http://api:8000/api/v1/ingest/fetch"
assert FETCH_TIMEOUT == 1800
assert DEFAULT_TIMEOUT == 30
print(f"\u2713 API_BASE_URL: {API_BASE_URL}")
print(f"\u2713 HEALTH_URL: {HEALTH_URL}")
print(f"\u2713 INGEST_FETCH_URL: {INGEST_FETCH_URL}")
print(f"\u2713 FETCH_TIMEOUT: {FETCH_TIMEOUT}s")

✓ API_BASE_URL: http://api:8000
✓ HEALTH_URL: http://api:8000/api/v1/ping
✓ INGEST_FETCH_URL: http://api:8000/api/v1/ingest/fetch
✓ FETCH_TIMEOUT: 1800s


In [3]:
from arxiv_ingestion.setup import setup_environment
from arxiv_ingestion.fetching import fetch_daily_papers
from arxiv_ingestion.reporting import generate_daily_report

assert callable(setup_environment)
assert callable(fetch_daily_papers)
assert callable(generate_daily_report)
print("\u2713 All task functions importable")

✓ All task functions importable


## 2. Ingestion Schemas

In [4]:
from src.schemas.api.ingest import IngestRequest, IngestResponse

# Valid request
req = IngestRequest(target_date="20260309")
assert req.target_date == "20260309"
print(f"\u2713 IngestRequest: target_date={req.target_date}")

# Default request (no date)
req_default = IngestRequest()
assert req_default.target_date is None
print(f"\u2713 IngestRequest default: target_date={req_default.target_date}")

# Response defaults
resp = IngestResponse()
assert resp.papers_fetched == 0
assert resp.arxiv_ids == []
print(f"\u2713 IngestResponse defaults: papers_fetched={resp.papers_fetched}, arxiv_ids={resp.arxiv_ids}")

# Invalid date format
from pydantic import ValidationError
try:
    IngestRequest(target_date="not-a-date")
    assert False, "Should have raised ValidationError"
except ValidationError:
    print("\u2713 IngestRequest rejects invalid date format")

✓ IngestRequest: target_date=20260309
✓ IngestRequest default: target_date=None
✓ IngestResponse defaults: papers_fetched=0, arxiv_ids=[]
✓ IngestRequest rejects invalid date format


## 3. Ingestion Endpoint (Mocked)

In [5]:
import asyncio
from unittest.mock import AsyncMock, MagicMock, patch
from pathlib import Path
import uuid

from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient

from src.routers.ingest import router
from src.dependency import get_db_session, get_paper_repository
from src.schemas.arxiv import ArxivPaper
from src.schemas.pdf import PDFContent, Section


async def test_ingest_endpoint():
    # Setup mocks
    mock_arxiv = AsyncMock()
    mock_arxiv.fetch_papers = AsyncMock(return_value=[
        ArxivPaper(
            arxiv_id="2602.00001", title="Test Paper",
            authors=["Alice"], abstract="Abstract",
            categories=["cs.AI"], published_date="2026-03-09T00:00:00Z",
            pdf_url="https://arxiv.org/pdf/2602.00001.pdf",
        )
    ])
    mock_arxiv.download_pdf = AsyncMock(return_value=Path("/tmp/test.pdf"))

    mock_parser = AsyncMock()
    mock_parser.parse_pdf = AsyncMock(return_value=PDFContent(
        raw_text="Full text", sections=[Section(title="Intro", content="Content")],
    ))

    mock_repo = AsyncMock()
    mock_paper = MagicMock(id=uuid.uuid4(), arxiv_id="2602.00001")
    mock_repo.upsert = AsyncMock(return_value=mock_paper)
    mock_repo.update_parsing_status = AsyncMock(return_value=mock_paper)

    mock_session = AsyncMock()

    app = FastAPI()
    app.include_router(router, prefix="/api/v1")
    app.dependency_overrides[get_db_session] = lambda: mock_session
    app.dependency_overrides[get_paper_repository] = lambda: mock_repo

    with (
        patch("src.routers.ingest.make_arxiv_client", return_value=mock_arxiv),
        patch("src.routers.ingest.make_pdf_parser_service", return_value=mock_parser),
    ):
        async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
            resp = await client.post("/api/v1/ingest/fetch", json={"target_date": "20260309"})

    data = resp.json()
    assert resp.status_code == 200
    assert data["papers_fetched"] == 1
    assert data["pdfs_downloaded"] == 1
    assert data["pdfs_parsed"] == 1
    assert "2602.00001" in data["arxiv_ids"]
    print(f"\u2713 Ingest endpoint: {data['papers_fetched']} fetched, {data['pdfs_parsed']} parsed")
    print(f"  arxiv_ids: {data['arxiv_ids']}")
    print(f"  errors: {data['errors']}")

await test_ingest_endpoint()

✓ Ingest endpoint: 1 fetched, 1 parsed
  arxiv_ids: ['2602.00001']
  errors: []


## 4. Task Function Tests (Mocked HTTP)

In [6]:
from datetime import datetime, UTC

# Test setup_environment
with patch("arxiv_ingestion.setup.httpx.get") as mock_get:
    mock_resp = MagicMock()
    mock_resp.json.return_value = {"status": "ok"}
    mock_resp.raise_for_status = MagicMock()
    mock_get.return_value = mock_resp
    setup_environment()
    print("\u2713 setup_environment: health check passed")

# Test fetch_daily_papers
with patch("arxiv_ingestion.fetching.httpx.post") as mock_post:
    ti = MagicMock()
    mock_resp = MagicMock()
    mock_resp.json.return_value = {"papers_fetched": 5, "arxiv_ids": ["123"]}
    mock_resp.raise_for_status = MagicMock()
    mock_post.return_value = mock_resp
    fetch_daily_papers(ti=ti, execution_date=datetime(2026, 3, 10, tzinfo=UTC))
    call_payload = mock_post.call_args[1]["json"]
    assert call_payload["target_date"] == "20260309"
    print(f"\u2713 fetch_daily_papers: target_date={call_payload['target_date']}")

# Test generate_daily_report
with patch("arxiv_ingestion.reporting.httpx.get") as mock_get:
    ti = MagicMock()
    ti.xcom_pull.return_value = {"papers_fetched": 5, "errors": []}
    mock_resp = MagicMock()
    mock_resp.json.return_value = {"status": "ok"}
    mock_resp.raise_for_status = MagicMock()
    mock_get.return_value = mock_resp
    generate_daily_report(ti=ti, execution_date=datetime(2026, 3, 10, tzinfo=UTC))
    print("\u2713 generate_daily_report: report logged successfully")

✓ setup_environment: health check passed
✓ fetch_daily_papers: target_date=20260309
✓ generate_daily_report: report logged successfully


## Summary

All S3.4 components verified:
- Constants and URLs configured correctly
- Task functions (setup, fetch, report) work with mocked HTTP
- Ingestion schemas validate input/output
- FastAPI endpoint orchestrates fetch -> parse -> store pipeline
- DAG configuration is in `airflow/dags/arxiv_paper_ingestion.py` (tested in Airflow container)